1. Loading the data

In [4]:
import urllib.request, os, random
import numpy as np
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED); np.random.seed(SEED)   # HMM is deterministic, but state the seed anyway

BASE = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/"
os.makedirs("data", exist_ok=True)
for split in ["train", "dev", "test"]:
    fn = f"en_ewt-ud-{split}.conllu"
    if not os.path.exists(f"data/{fn}"):
        urllib.request.urlretrieve(BASE + fn, f"data/{fn}")

def read_conllu(path):
    """Return list of (tokens, upos_tags), one per sentence."""
    sents, words, tags = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("#"):
                continue
            if not line:                                   # blank line = end of sentence
                if words:
                    sents.append((words, tags)); words, tags = [], []
                continue
            cols = line.split("\t")
            if "-" in cols[0] or "." in cols[0]:           # skip multiword ranges / empty nodes
                continue
            words.append(cols[1]); tags.append(cols[3])    # FORM, UPOS
    if words:
        sents.append((words, tags))
    return sents

train = read_conllu("data/en_ewt-ud-train.conllu")
dev   = read_conllu("data/en_ewt-ud-dev.conllu")
test  = read_conllu("data/en_ewt-ud-test.conllu")
tag_names = sorted({t for _, ts in train for t in ts})
print(len(train), len(dev), len(test), len(tag_names))
print(tag_names)

12544 2001 2077 17
['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


2. The HMM tagger

In [5]:
OPEN = {"ADJ", "ADV", "INTJ", "NOUN", "PROPN", "VERB", "X"}

def suffix_key(w, max_n=3):
    """Candidate (suffix, is_capitalized) keys, longest suffix first."""
    lw, cap = w.lower(), w[0].isupper()
    return [(lw[-n:], cap) for n in range(min(max_n, len(lw)), 0, -1)]

class HMMTagger:
    def __init__(self, lam=0.1, rare_thresh=5):
        self.lam, self.rare = lam, rare_thresh

    def fit(self, data):
        lam = self.lam
        self.tags = sorted({t for _, ts in data for t in ts})
        K = len(self.tags); ti = {t: i for i, t in enumerate(self.tags)}
        self.K = K
        trans = np.zeros((K + 1, K + 1))            # rows: previous tag (K = START); cols: next tag (K = END)
        emit = defaultdict(lambda: np.zeros(K))     # word -> counts per tag
        tag_count = np.zeros(K)
        word_freq = Counter(w.lower() for ws, _ in data for w in ws)
        suf = defaultdict(lambda: np.zeros(K))      # (suffix, cap) -> counts per tag
        for ws, ts in data:
            prev = K
            for w, t in zip(ws, ts):
                i = ti[t]
                trans[prev, i] += 1
                emit[w.lower()][i] += 1
                tag_count[i] += 1
                if word_freq[w.lower()] <= self.rare:       # rare words train the suffix model
                    for key in suffix_key(w):
                        suf[key][i] += 1
                prev = i
            trans[prev, K] += 1
        # add-lambda smoothed log-probabilities
        self.log_trans = np.log((trans + lam) / (trans.sum(1, keepdims=True) + lam * (K + 1)))
        V = len(emit)
        self.emit = {w: np.log((c + lam) / (tag_count + lam * (V + 1))) for w, c in emit.items()}
        self.suf = {k: np.log((c + lam) / (tag_count + lam * len(suf))) for k, c in suf.items()}
        open_mask = np.array([t in OPEN for t in self.tags])
        self.unif = np.where(open_mask, np.log(1 / open_mask.sum()), np.log(1e-8))

    def emission(self, w):
        lw = w.lower()
        if lw in self.emit:                       # known word
            return self.emit[lw]
        for key in suffix_key(w):                 # unknown: longest seen suffix + capitalization
            if key in self.suf:
                return self.suf[key]
        return self.unif                          # fallback: uniform over open-class tags

    def tag(self, words):
        """Viterbi decoding in log-space."""
        n, K = len(words), self.K
        E = np.array([self.emission(w) for w in words])
        delta = np.full((n, K), -np.inf); back = np.zeros((n, K), int)
        delta[0] = self.log_trans[K, :K] + E[0]
        for i in range(1, n):
            sc = delta[i - 1][:, None] + self.log_trans[:K, :K]
            back[i] = sc.argmax(0)
            delta[i] = sc.max(0) + E[i]
        best = (delta[-1] + self.log_trans[:K, K]).argmax()
        path = [best]
        for i in range(n - 1, 0, -1):
            best = back[i][best]; path.append(best)
        return [self.tags[i] for i in reversed(path)]

def accuracy(model, data):
    c = n = 0
    for ws, ts in data:
        pred = model.tag(ws)
        c += sum(p == g for p, g in zip(pred, ts)); n += len(ts)
    return c / n

In [6]:
# 3. The main result

In [7]:
hmm = HMMTagger(lam=0.1)
hmm.fit(train)
print("HMM dev accuracy :", accuracy(hmm, dev))
print("HMM test accuracy:", accuracy(hmm, test))

HMM dev accuracy : 0.9125974232543344
HMM test accuracy: 0.9191838686538615


4. Analysis for the report

In [8]:
train_vocab = {w.lower() for ws, _ in train for w in ws}
known_c = known_n = unk_c = unk_n = 0
confusions = Counter(); errors = []

for ws, gold in test:
    pred = hmm.tag(ws)
    for w, g, p in zip(ws, gold, pred):
        ok = (g == p)
        if w.lower() in train_vocab: known_n += 1; known_c += ok
        else:                        unk_n += 1;   unk_c += ok
        if not ok: confusions[(g, p)] += 1
    if pred != gold and len(errors) < 200:
        errors.append((ws, gold, pred))

print(f"Overall : {(known_c+unk_c)/(known_n+unk_n):.4f}")
print(f"Known   : {known_c/known_n:.4f}  ({known_n} tokens)")
print(f"Unknown : {unk_c/unk_n:.4f}  ({unk_n} tokens, {unk_n/(known_n+unk_n):.1%} of test)")
print("\nTop 10 confusions (gold -> predicted):")
for (g, p), c in confusions.most_common(10):
    print(f"  {g:6s} -> {p:6s} {c}")

for ws, gold, pred in errors[:3]:                 # example sentences for qualitative analysis
    print()
    for w, g, p in zip(ws, gold, pred):
        print(f"{w:15s} gold={g:6s} pred={p:6s} {'' if g==p else '<-- WRONG'}")

Overall : 0.9192
Known   : 0.9321  (23212 tokens)
Unknown : 0.7598  (1882 tokens, 7.5% of test)

Top 10 confusions (gold -> predicted):
  PROPN  -> NOUN   288
  NOUN   -> PROPN  151
  VERB   -> AUX    105
  VERB   -> NOUN   87
  SCONJ  -> ADP    79
  NOUN   -> VERB   78
  NOUN   -> ADJ    73
  ADV    -> ADP    55
  ADV    -> ADJ    54
  ADJ    -> VERB   51

What            gold=PRON   pred=PRON   
if              gold=SCONJ  pred=SCONJ  
Google          gold=PROPN  pred=PROPN  
Morphed         gold=VERB   pred=PROPN  <-- WRONG
Into            gold=ADP    pred=ADP    
GoogleOS        gold=PROPN  pred=PROPN  
?               gold=PUNCT  pred=PUNCT  

What            gold=PRON   pred=PRON   
if              gold=SCONJ  pred=SCONJ  
Google          gold=PROPN  pred=PROPN  
expanded        gold=VERB   pred=VERB   
on              gold=ADP    pred=ADP    
its             gold=PRON   pred=PRON   
search          gold=NOUN   pred=NOUN   
-               gold=PUNCT  pred=PUNCT  
engine         